# Standalone Orbital Visualiser

### Calculation of Cubes

In [1]:
# Calculation Dependencies
from aiida.engine import run_get_node
from aiida.orm import FolderData

from topworkchain import PrototypeTopWorkChain
from aiida import load_profile

In [2]:
# Load AiiDA profile
load_profile()

Profile<uuid='52dacf545d5c4d089ba5e0dd6267dcf0' name='presto'>

In [3]:
# Run the workchain that calculates NTOs, converts them to cube then compresses and returns them as FolderData
results, node = run_get_node(PrototypeTopWorkChain)

# Load the FolderData node containing the compressed cube files
cube_folder = results["cube_folder"]

# Load the Dict node containing all of the relevant details of the transitions.
transition_info_node = results["transition_info"]

# Convert to a standard Python dict.
transition_info = transition_info_node.get_dict()

04/25/2026 08:31:32 PM <326862> aiida.broker.rabbitmq: [WARNING] RabbitMQ v3.12.1 is not supported and will cause unexpected problems!
04/25/2026 08:31:32 PM <326862> aiida.broker.rabbitmq: [WARNING] It can cause long-running workflows to crash and jobs to be submitted multiple times.
04/25/2026 08:31:32 PM <326862> aiida.broker.rabbitmq: [WARNING] See https://github.com/aiidateam/aiida-core/wiki/RabbitMQ-version-to-use for details.


uuid: e7ce30ae-4e74-4f91-a76a-10f85bcd820f (pk: 35548) (subworkchains.OrcaWorkChain)
uuid: a5bcb1ea-e662-4d5c-8af1-c1f220c91194 (pk: 35573)
uuid: 69f501fe-b9d3-43c1-a82d-912a17a5db40 (pk: 35587)
uuid: b9c5ab91-7367-4f44-ae1f-ef12733a80c1 (pk: 35602)
uuid: f88021c6-4ab4-4e9a-9fbb-96b52b08f1cf (pk: 35616)
uuid: 764f5c2b-59c0-46e7-9f57-4e8a0c07d112 (pk: 35631)
uuid: 12cf48a1-7cc1-4248-9363-93f1c9544105 (pk: 35645)
uuid: abb7d928-d900-4c74-9252-eb8a158d725c (pk: 35659)
uuid: b8affe7a-0c07-465b-858d-1638d41c7d09 (pk: 35673)
uuid: 03db0e7e-1984-42c8-8c2f-c6b27b2edb91 (pk: 35688)
uuid: 6193fc11-fabb-48e8-a45c-4be601844aa8 (pk: 35702)
uuid: 4aad177b-df3b-4c6a-8805-b8d4a881566f (pk: 35717)
uuid: 6dacb15f-45be-4de0-93b1-bcfbbc8018fc (pk: 35731)
uuid: c5003f80-c187-400e-bc4d-6e5937023271 (pk: 35746)
uuid: 43c3657c-ba69-4128-bb01-64bd85d05d92 (pk: 35760)
uuid: 667438c3-51c1-494e-a9a4-4cb3b3ed552b (pk: 35774)
uuid: 4027255f-7211-440b-98a6-af419fb75f77 (pk: 35788)
uuid: f006fe75-c19c-440a-9d83-a6ae3

### Visualisation

In [4]:
import nglview as nv
import ipywidgets as widgets
from ipywidgets import Layout

import ase.io
from pathlib import Path

In [8]:
#----CUBE FILE HANDLING-----
# Check if local directory exists, if it does clear as cube files can be pretty big and there are a lot.
# If not directory, create one

output_dir = Path("extracted_cubes")

if output_dir.is_dir():
    for item in output_dir.iterdir():
        item.unlink()
else:
    output_dir.mkdir(exist_ok=True)

# Create a list of the cube files in the data node
names = cube_folder.list_object_names()

# Read in the cube files and write them to temporary files
for name in names:
    with cube_folder.open(name, mode="rb") as file_in:
        with open(f"{output_dir}/{name}.cube", "wb") as file_out:
            file_out.write(file_in.read())


# ----EXCITED STATE DROPDOWN----
specific_excitations = list(transition_info.keys())

# Create dropdown widgets for selecting the cube files
excitation_dropdown = widgets.Dropdown(
    options=specific_excitations,
    value=specific_excitations[0] if specific_excitations else None,
    description="Excitation (s)"
 )


# ----TRANSITION STATE DROPDOWN----
#Function to change the options available in the transition_dropdown.
def transition_update(change):
    #Isolate specific relevant transitions.
    specific_transitions = transition_info[excitation_dropdown.value]
    #Adjust format of transitions.
    specific_transitions2 = []
    for each_transition in specific_transitions:
        specific_transitions2.append((((each_transition[0])[0]+" -> "+(each_transition[0])[1]+" ("+each_transition[1]+")"), each_transition))
    transition_dropdown.options = specific_transitions2

transition_dropdown = widgets.Dropdown(
    description="Transition"
 )

#Run transition_update when excitation_dropdown is clicked.
excitation_dropdown.observe(transition_update, names="value")

#Initialise the transition_dropdown with initial options and value.
transition_update({"new": excitation_dropdown.value})
transition_dropdown.value = transition_dropdown.options[0][1] if transition_dropdown.options else None


# ----NGL VIEWER----
# Initialize views with current dropdown selections
CUBE_PATH_1 = f"./extracted_cubes/{"s"+excitation_dropdown.value+"_"+((transition_dropdown.value[0])[0])[:-1]+".cube"}"
CUBE_PATH_2 = f"./extracted_cubes/{"s"+excitation_dropdown.value+"_"+((transition_dropdown.value[0])[1])[:-1]+".cube"}"

# Create NGL views for the selected cube files
molecule1 = nv.show_ase(ase.io.read(CUBE_PATH_1))
molecule2 = nv.show_ase(ase.io.read(CUBE_PATH_2))
NTO_1 = molecule1.add_component(CUBE_PATH_1, ext="cube")
NTO_2 = molecule2.add_component(CUBE_PATH_2, ext="cube")


# ----ISOVALUES AND COLORS----
# Create sliders for adjusting the isovalue (not visible in the NGL view, but used to set the isovalue of the surfaces)
positive_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="+ isovalue",
    readout_format=".1e",
 )
negative_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="- isovalue",
    readout_format=".1e",
 )

# List of colour blind friendly colours
colours = ['#377eb8', '#ff7f00', '#4daf4a',
                  '#f781bf', '#a65628', '#984ea3',
                  '#999999', '#e41a1c', '#dede00']

# To store colour buttons
pos_palette_buttons = []
neg_palette_buttons = []

def set_positive_colour(colour):
    selected_colours["positive"] = colour
    redraw_surfaces()

def set_negative_colour(colour):
    selected_colours["negative"] = colour
    redraw_surfaces()

# Create buttons for each colour for both palettes
for c in colours:
    b = widgets.Button(
        layout=widgets.Layout(width='30px', height='30px'),
        style={'button_color': c}
    )
    b.on_click(lambda _, c=c: set_positive_colour(c))
    pos_palette_buttons.append(b)

for c in colours:
    b = widgets.Button(
        layout=widgets.Layout(width='30px', height='30px'),
        style={'button_color': c}
    )
    b.on_click(lambda _, c=c: set_negative_colour(c))
    neg_palette_buttons.append(b)

pos_palette = widgets.HBox(pos_palette_buttons)
neg_palette = widgets.HBox(neg_palette_buttons)

# Preset colours 
selected_colours = {
    "positive": "#377eb8",
    "negative": "#e41a1c"
}

# Create a box to contain the colour palettes
palette_box = widgets.VBox([
    widgets.HTML("<b>Positive colour</b>"),
    pos_palette,
    widgets.HTML("<b>Negative colour</b>"),
    neg_palette
    
])

# Create drop down for displaying colours
accordion = widgets.Accordion(children=[palette_box])
accordion.set_title(0, 'Select colours')
accordion.selected_index = None  # start closed
accordion.layout = Layout(width="25%")

status = widgets.HTML()

# Define functions to redraw the surfaces when the sliders or colour scheme are changed
def redraw_NTO_1(_=None):
    NTO_1.clear()
    NTO_1.add_surface(
        color=selected_colours["positive"],
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_1.add_surface(
        color=selected_colours["negative"],
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_NTO_2(_=None):
    NTO_2.clear()
    NTO_2.add_surface(
        color=selected_colours["positive"],
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_2.add_surface(
        color=selected_colours["negative"],
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_surfaces(_=None):
    redraw_NTO_1()
    redraw_NTO_2()

def update_NTO_1(_=None):
    global NTO_1
    NTO_1.clear()
    NTO_1 = molecule1.add_component(f"./extracted_cubes/{"s"+excitation_dropdown.value+"_"+((transition_dropdown.value[0])[0])[:-1]+".cube"}", ext="cube")  # Add path!
    redraw_NTO_1()
    
def update_NTO_2(_=None):
    global NTO_2
    NTO_2.clear()
    NTO_2 = molecule2.add_component(f"./extracted_cubes/{"s"+excitation_dropdown.value+"_"+((transition_dropdown.value[0])[1])[:-1]+".cube"}", ext="cube")  # Add path!
    redraw_NTO_2()

# Set up observers to redraw the surfaces when the sliders or colour scheme are changed
for control in [positive_level, negative_level]:
    control.observe(redraw_surfaces, names="value")

excitation_dropdown.observe(update_NTO_1, names="value")
transition_dropdown.observe(update_NTO_1, names="value")
excitation_dropdown.observe(update_NTO_2, names="value")
transition_dropdown.observe(update_NTO_2, names="value")

redraw_surfaces()

# Arrange the widgets in a layout
controls = widgets.VBox(
    [
        widgets.HTML("<b>Cube isosurfaces</b>"),
        accordion,
        status,
    ],
 )
molecule1_box = widgets.VBox([excitation_dropdown, molecule1], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule2_box = widgets.VBox([transition_dropdown, molecule2], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule_box = widgets.HBox([molecule1_box, molecule2_box], layout=Layout(width="100%", flex="1 1 100%"))

widgets.VBox([controls, molecule_box], layout = Layout(width="100%", flex="1 1 100%"))
